# SIMULADOR DE APROBACIÓN DE PRÉSTAMOS — Fintech Credit Scoring

## Versión: 1.0 | Autor: Agustín Musanti

In [5]:
from IPython.display import display, HTML

# ── PARÁMETROS DE ENTRADA ────────────────────────────────────────
ingreso          = 1_200_000
deuda_actual     = 180_000
monto_solicitado = 2_500_000
plazo_meses      = 24

# ── TASA NOMINAL ANUAL (fija, definida aquí) ─────────────────────
TNA = 0.60   # 60% nominal anual → 5% mensual


#  FUNCIONES

def calcular_cuota(monto: float, tna: float, plazo: int) -> float:
    """Sistema francés con tasa mensual derivada de la TNA."""
    tasa_mensual = tna / 12
    if tasa_mensual == 0:
        return monto / plazo
    return monto * tasa_mensual / (1 - (1 + tasa_mensual) ** -plazo)


def calcular_score(ingreso: float, deuda_actual: float,
                   cuota: float, monto: float) -> int:
    """
    Score 0–100 basado en tres reglas simples ponderadas:
      - Cuota / Ingreso     → peso 50 pts
      - Deuda actual / Ingreso → peso 30 pts
      - Monto / Ingreso     → peso 20 pts
    """
    score = 0

    r_cuota = cuota / ingreso if ingreso > 0 else 1
    if r_cuota <= 0.25:   score += 50
    elif r_cuota <= 0.35: score += 35
    elif r_cuota <= 0.45: score += 18
    else:                 score += 0

    r_deuda = deuda_actual / ingreso if ingreso > 0 else 1
    if r_deuda <= 0.20:   score += 30
    elif r_deuda <= 0.35: score += 18
    elif r_deuda <= 0.50: score += 8
    else:                 score += 0

    r_monto = monto / ingreso if ingreso > 0 else 99
    if r_monto <= 2:      score += 20
    elif r_monto <= 4:    score += 12
    elif r_monto <= 6:    score += 5
    else:                 score += 0

    return min(max(int(score), 0), 100)


def fmt_pesos(v: float) -> str:
    return "$ " + f"{int(round(v)):,}".replace(",", ".")

def fmt_pct(v: float) -> str:
    return f"{int(round(v * 100))} %"

def fmt_veces(v: float) -> str:
    r = round(v, 1)
    return f"{int(r)}×" if r == int(r) else f"{r}×"


def renderizar_panel(ingreso, deuda_actual, monto_solicitado,
                     plazo_meses, tna=TNA):

    tasa_mensual = tna / 12
    cuota  = calcular_cuota(monto_solicitado, tna, plazo_meses)
    score  = calcular_score(ingreso, deuda_actual, cuota, monto_solicitado)

    r_cuota = cuota / ingreso if ingreso > 0 else 0
    r_deuda = deuda_actual / ingreso if ingreso > 0 else 0
    r_monto = monto_solicitado / ingreso if ingreso > 0 else 0

    tna_display = f"{int(round(tna * 100))} % TNA"
    tm_display  = f"{round(tasa_mensual * 100, 1):.1f} % mensual".replace(".", ",")

    # ── Estado y colores ────────────────────────────────────────
    if score >= 65:
        estado      = "APROBADO"
        b_bg        = "#dcfce7"
        b_color     = "#166534"
        b_border    = "#86efac"
        score_color = "#16a34a"
        dot_color   = "#22c55e"
        razon = (
            f"La cuota representa el {fmt_pct(r_cuota)} del ingreso mensual "
            f"y el endeudamiento previo es del {fmt_pct(r_deuda)}. "
            f"Ambos indicadores están dentro de los rangos aceptables."
        )
    elif score >= 40:
        estado      = "EN REVISIÓN"
        b_bg        = "#fef9c3"
        b_color     = "#854d0e"
        b_border    = "#fde047"
        score_color = "#d97706"
        dot_color   = "#f59e0b"
        razon = (
            f"La cuota equivale al {fmt_pct(r_cuota)} del ingreso "
            f"y el monto solicitado es {fmt_veces(r_monto)} el ingreso mensual. "
            f"Se recomienda análisis manual antes de aprobar."
        )
    else:
        estado      = "RECHAZADO"
        b_bg        = "#fee2e2"
        b_color     = "#991b1b"
        b_border    = "#fca5a5"
        score_color = "#dc2626"
        dot_color   = "#ef4444"
        razon = (
            f"La cuota ({fmt_pct(r_cuota)} del ingreso) o el nivel de "
            f"endeudamiento ({fmt_pct(r_deuda)}) superan los límites permitidos."
        )

    # ── Posición del marcador en la barra (0–100%) ──────────────
    marker_left = score  # igual al score en %

    html = f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap');

      .lc-wrap {{
        font-family: 'Plus Jakarta Sans', sans-serif;
        background: #ffffff;
        border: 1px solid #e2e8f0;
        border-radius: 20px;
        padding: 36px 40px 30px;
        max-width: 660px;
        margin: 24px auto;
        box-shadow: 0 2px 16px rgba(15,23,42,.08);
        color: #0f172a;
      }}

      /* HEADER */
      .lc-header {{
        display: flex;
        justify-content: space-between;
        align-items: flex-start;
        margin-bottom: 30px;
        padding-bottom: 22px;
        border-bottom: 1px solid #f1f5f9;
      }}
      .lc-eyebrow {{
        font-size: 10.5px;
        font-weight: 700;
        letter-spacing: .14em;
        text-transform: uppercase;
        color: #94a3b8;
        margin-bottom: 5px;
      }}
      .lc-loan-amount {{
        font-size: 28px;
        font-weight: 800;
        color: #0f172a;
        letter-spacing: -.5px;
        margin-bottom: 4px;
      }}
      .lc-loan-meta {{
        font-size: 13px;
        color: #64748b;
        font-weight: 400;
      }}
      .lc-loan-meta strong {{
        color: #334155;
        font-weight: 600;
      }}
      .lc-badge {{
        display: flex;
        align-items: center;
        gap: 7px;
        font-size: 11.5px;
        font-weight: 700;
        letter-spacing: .10em;
        text-transform: uppercase;
        padding: 7px 16px;
        border-radius: 999px;
        border: 1.5px solid {b_border};
        background: {b_bg};
        color: {b_color};
        white-space: nowrap;
        margin-top: 2px;
      }}
      .lc-dot {{
        width: 7px; height: 7px;
        border-radius: 50%;
        background: {dot_color};
        flex-shrink: 0;
      }}

      /* METRIC CARDS */
      .lc-cards {{
        display: grid;
        grid-template-columns: repeat(3, 1fr);
        gap: 10px;
        margin-bottom: 28px;
      }}
      .lc-card {{
        background: #f8fafc;
        border: 1px solid #f1f5f9;
        border-radius: 12px;
        padding: 14px 14px 13px;
      }}
      .lc-card-label {{
        font-size: 10px;
        font-weight: 700;
        letter-spacing: .09em;
        text-transform: uppercase;
        color: #94a3b8;
        margin-bottom: 5px;
      }}
      .lc-card-value {{
        font-family: 'JetBrains Mono', monospace;
        font-size: 17px;
        font-weight: 500;
        color: #0f172a;
        line-height: 1;
      }}
      .lc-card-sub {{
        font-size: 10.5px;
        color: #94a3b8;
        margin-top: 4px;
        font-weight: 400;
      }}

      /* SCORE */
      .lc-score-section {{
        display: flex;
        align-items: center;
        gap: 24px;
        margin-bottom: 22px;
        background: #f8fafc;
        border: 1px solid #f1f5f9;
        border-radius: 14px;
        padding: 20px 22px;
      }}
      .lc-score-num {{
        font-family: 'JetBrains Mono', monospace;
        font-size: 56px;
        font-weight: 700;
        color: {score_color};
        line-height: 1;
        min-width: 78px;
        text-align: center;
      }}
      .lc-score-right {{
        flex: 1;
      }}
      .lc-score-label {{
        font-size: 10.5px;
        font-weight: 700;
        letter-spacing: .11em;
        text-transform: uppercase;
        color: #94a3b8;
        margin-bottom: 10px;
      }}
      .lc-bar-track {{
        background: #e2e8f0;
        border-radius: 999px;
        height: 9px;
        position: relative;
      }}
      .lc-bar-fill {{
        height: 100%;
        width: {score}%;
        background: linear-gradient(90deg, {score_color}88, {score_color});
        border-radius: 999px;
      }}
      .lc-bar-marker {{
        position: absolute;
        top: 50%;
        left: {marker_left}%;
        transform: translate(-50%, -50%);
        width: 15px; height: 15px;
        border-radius: 50%;
        background: {score_color};
        border: 2.5px solid #fff;
        box-shadow: 0 0 0 1.5px {score_color}55;
      }}
      .lc-bar-zones {{
        display: flex;
        justify-content: space-between;
        margin-top: 7px;
      }}
      .lc-zone {{
        font-size: 9.5px;
        color: #cbd5e1;
        font-family: 'JetBrains Mono', monospace;
      }}
      .lc-zone.active {{
        color: {score_color};
        font-weight: 600;
      }}

      /* RAZON */
      .lc-reason {{
        font-size: 13px;
        color: #475569;
        line-height: 1.65;
        border-top: 1px solid #f1f5f9;
        padding-top: 18px;
      }}
      .lc-reason-icon {{
        display: inline-block;
        margin-right: 5px;
        opacity: .6;
      }}

      /* FOOTER */
      .lc-footer {{
        margin-top: 16px;
        font-size: 10px;
        color: #cbd5e1;
        text-align: right;
        letter-spacing: .04em;
      }}
    </style>

    <div class="lc-wrap">

      <!-- HEADER -->
      <div class="lc-header">
        <div>
          <div class="lc-eyebrow">Préstamo solicitado</div>
          <div class="lc-loan-amount">{fmt_pesos(monto_solicitado)}</div>
          <div class="lc-loan-meta">
            <strong>{plazo_meses} meses</strong>
            &nbsp;·&nbsp;
            Tasa: <strong>{tna_display}</strong>
            &nbsp;({tm_display})
          </div>
        </div>
        <div class="lc-badge">
          <span class="lc-dot"></span>
          {estado}
        </div>
      </div>

      <!-- METRIC CARDS -->
      <div class="lc-cards">
        <div class="lc-card">
          <div class="lc-card-label">Cuota mensual</div>
          <div class="lc-card-value">{fmt_pesos(cuota)}</div>
          <div class="lc-card-sub">estimada</div>
        </div>
        <div class="lc-card">
          <div class="lc-card-label">Cuota / Ingreso</div>
          <div class="lc-card-value">{fmt_pct(r_cuota)}</div>
          <div class="lc-card-sub">límite sugerido: 35 %</div>
        </div>
        <div class="lc-card">
          <div class="lc-card-label">Deuda / Ingreso</div>
          <div class="lc-card-value">{fmt_pct(r_deuda)}</div>
          <div class="lc-card-sub">límite sugerido: 40 %</div>
        </div>
        <div class="lc-card">
          <div class="lc-card-label">Ingreso mensual</div>
          <div class="lc-card-value">{fmt_pesos(ingreso)}</div>
          <div class="lc-card-sub">declarado</div>
        </div>
        <div class="lc-card">
          <div class="lc-card-label">Deuda actual</div>
          <div class="lc-card-value">{fmt_pesos(deuda_actual)}</div>
          <div class="lc-card-sub">mensual</div>
        </div>
        <div class="lc-card">
          <div class="lc-card-label">Monto / Ingreso</div>
          <div class="lc-card-value">{fmt_veces(r_monto)}</div>
          <div class="lc-card-sub">límite sugerido: 4×</div>
        </div>
      </div>

      <!-- SCORE -->
      <div class="lc-score-section">
        <div class="lc-score-num">{score}</div>
        <div class="lc-score-right">
          <div class="lc-score-label">Score de aprobación &nbsp;·&nbsp; modelo simplificado de scoring</div>
          <div class="lc-bar-track">
            <div class="lc-bar-fill"></div>
            <div class="lc-bar-marker"></div>
          </div>
          <div class="lc-bar-zones">
            <span class="lc-zone">0</span>
            <span class="lc-zone {'active' if score < 40 else ''}">Rechazado &lt;40</span>
            <span class="lc-zone {'active' if 40 <= score < 65 else ''}">Revisión 40–64</span>
            <span class="lc-zone {'active' if score >= 65 else ''}">Aprobado ≥65</span>
            <span class="lc-zone">100</span>
          </div>
        </div>
      </div>

      <!-- RAZÓN -->
      <div class="lc-reason">
        <span class="lc-reason-icon">💬</span>{razon}
      </div>

      <div class="lc-footer">Simulación educativa · No constituye oferta crediticia</div>

    </div>
    """

    display(HTML(html))


# ── EJECUTAR ─────────────────────────────────────────────────────
renderizar_panel(
    ingreso          = ingreso,
    deuda_actual     = deuda_actual,
    monto_solicitado = monto_solicitado,
    plazo_meses      = plazo_meses,
)